# 🌪️ StormCost v3 — Predicting Property Damage from Storm Narratives
**Competition:** StormCost on Kaggle · hosted by BTAJI Crew Community
**Approach:** frozen contextual sentence encoder + LightGBM **and** a fine-tuned MiniLM transformer, blended out-of-fold, with era-aware, metric-aware post-processing.

## 1. Problem in plain words
Each row is a severe-weather incident (hail, tornado, flood, ...). We get a short narrative written by an officer plus structured fields (state, year, magnitude, injuries...). We must predict **property damage in US dollars**.

The score has two equally weighted parts:

| Part | Question | Metric |
|---|---|---|
| Occurrence | Did any damage happen? | `2·balanced_accuracy − 1` (predicted 0 = "no damage") |
| Magnitude | If yes, how much? | `Σ min(ŷ,y) / Σ max(ŷ,y)` over truly damaged rows (dollar space, so big events dominate) |

`final = 1 + 99 · (0.5·occurrence + 0.5·magnitude)` (all-zeros = 1, perfect = 100)

## 2. Solution overview
```
                ┌─ frozen MiniLM → PCA ─┐
narratives ─────┤                       ├─► LightGBM classifier → P(damage)      ─┐
structured ─────┘   + structured fields ┘   LightGBM regressor  → log(damage|dmg) │
                                                                                  ├─► blend (weights chosen out-of-fold)
narratives + context text ─► fine-tuned MiniLM (2 heads) → P(damage), log(damage) ┘
                                                                                  │
         P(damage) > era-specific threshold ?  expm1(log-pred) × scale  :  exactly 0
```
**Key design choices (and why):**
1. **Two-head modelling** – the metric is two-headed, so we predict occurrence and magnitude separately.
2. **Era awareness** – train is 1997-2005 (3% of rows, ~86% damaged) + 2007-2016 (~36% damaged). Test adds 1993-95 (no train years, no episode narratives) and 2017-2025. The old reporting era is ~6.8× more common in test than in train, so we use an *old-era flag*, *separate thresholds per era* and *re-weight old-era rows* when tuning.
3. **Tail-weighted regression** – the top 1% of damaged rows hold ~88% of the dollars, so the regressor's sample weight is `damage ** 0.2` (a gentle push toward getting big events right).
4. **Metric-aware post-processing** – a log-trained regressor under-predicts the tail, but the score is in dollars. We tune the decision thresholds and a global dollar scale on out-of-fold predictions.
5. **Safety first** – a GBDT-only `submission.csv` is written *immediately*; the transformer stage can only replace it if it beats it out-of-fold. Hard wall-clock guards keep the run inside the time limit.
6. **Rule-compliant text modelling** – only dense contextual encoders (frozen or fine-tuned). No TF-IDF / bag-of-words / n-gram counters.

## 3. Competition-rules compliance
| Rule | How this notebook complies |
|---|---|
| Text representation must be a learned contextual encoder | `sentence-transformers/all-MiniLM-L6-v2`, frozen (GBDT branch) and fine-tuned (NN branch) |
| No TF-IDF, BoW, n-gram counters, one-hot token matrices, averaged static embeddings | None used |
| `episode_group` / `sample_id` not model features | `episode_group` only builds CV folds; both are asserted absent from the feature matrix |
| Group-size features | `USE_GROUP_SIZE=False` (switch on only if the host confirms it is allowed for test) |
| Don't train on the `-1.0` placeholder | Test labels never used (asserted) |
| Grouped CV; don't tune to the public LB | `GroupKFold` by `episode_group`; the public LB is never consulted |
| Predictions finite & non-negative, one row per `sample_id`, exact 0.0 for "no damage" | Validated before and after saving |
| Runtime: ≤ 1.5 h single GPU, training included | Hard time guards; total runtime is logged and asserted at the end |
| No external data / database lookup | None used (only the pretrained encoder weights) |

**How to run (Kaggle):** add the *StormCost* competition data → Settings → Accelerator **GPU T4 x2** → Internet **on** (the encoder weights are downloaded from the HF Hub) → *Run all*.

## 4. Setup & configuration
**In simple words:** import the libraries and put every adjustable setting in one place (`CFG`) so the experiment is reproducible. A small stopwatch (`tick`) prints elapsed minutes, because the competition has a runtime limit. If no GPU is found we stop early, since embedding ~590k texts on CPU would take hours.

In [1]:
import os, time, gc, warnings, traceback
import numpy as np, pandas as pd, torch
import lightgbm as lgb
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
warnings.filterwarnings("ignore")
T0 = time.time()
def tick(msg): print(f"[{(time.time()-T0)/60:5.1f} min] {msg}", flush=True)
def elapsed_min(): return (time.time() - T0) / 60

CFG = dict(
    ENCODER="sentence-transformers/all-MiniLM-L6-v2",
    EV_LEN=64, EP_LEN=128,          # frozen-encoder max tokens (event text averages ~40 tokens)
    PCA_EV=32, PCA_EP=24,
    N_FOLDS=3, SEED=42,
    BUDGET_MIN=78,                  # everything must be finished (submission written) by this wall-clock minute; hard limit is 90
    TAIL_ALPHA=0.2,                 # regressor sample weight = damage ** alpha
    USE_GROUP_SIZE=False,           # only enable if the host confirms group-size features are allowed for test
    # fine-tuned transformer
    NN_ENABLE=True, NN_FAKE=False,  # NN_FAKE is only for offline smoke tests
    NN_MAXLEN=96, NN_BS=128, NN_LR=6e-5, NN_FOLD_TRAIN_SEC=480,
    ALLOW_CPU=False, SAMPLE=None,   # SAMPLE=20000 -> quick smoke test on a subset
)
np.random.seed(CFG["SEED"])
device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)
if device == "cpu" and not CFG["ALLOW_CPU"]:
    raise RuntimeError("No GPU detected. Settings > Accelerator > GPU T4 x2, then restart. (Embedding on CPU would take hours.)")

DEVICE: cuda

## 5. Load the data
**In simple words:** find the competition files on Kaggle's disk and read train and test. (`SAMPLE` lets you smoke-test the whole notebook on a small subset.)

In [2]:
cands = ["/kaggle/input/competitions/storm-cost", "/kaggle/input/storm-cost"]
DATA_DIR = next(Path(p) for p in cands if Path(p, "train.csv").exists())
tr = pd.read_csv(DATA_DIR / "train.csv"); te = pd.read_csv(DATA_DIR / "test.csv")
if CFG["SAMPLE"]:
    keep = tr.episode_group.drop_duplicates().sample(frac=0.15, random_state=0)
    tr = tr[tr.episode_group.isin(keep)].reset_index(drop=True); te = te.sample(CFG["SAMPLE"], random_state=0).reset_index(drop=True)
ntr = len(tr)
tick(f"train {tr.shape} test {te.shape} from {DATA_DIR}")

[  0.3 min] train (358976, 20) test (229965, 20) from /kaggle/input/competitions/storm-cost

## 6. Data audit (sanity checks)
**In simple words:** before modelling, make sure the data is what the competition says it is, and measure the **era shift** that drives several later decisions. If an `assert` fails, something is wrong with the download.

In [ ]:
TARGET = "damage_property_usd"
expected = {"sample_id","episode_group","year","month_name","state","event_type","lon","lat","injuries_direct",
            "injuries_indirect","deaths_direct","deaths_indirect","magnitude","magnitude_type","tor_f_scale",
            "tor_length","tor_width","event_narrative_redacted","episode_narrative_redacted"}
assert expected <= set(tr.columns) and expected <= set(te.columns), "missing columns"
assert (tr[TARGET] >= 0).all(), "negative labels in train?"
assert (te[TARGET] == -1.0).all(), "test placeholder should be -1.0 (and we never use it)"
assert tr.sample_id.is_unique and te.sample_id.is_unique

print("✅ schema OK | episode groups shared by train and test:", len(set(tr.episode_group) & set(te.episode_group)))
print(f"Old era (<=2005): train share {(tr.year <= 2005).mean():.1%} | test share {(te.year <= 2005).mean():.1%}")
print(f"Damage rate in train: old era {(tr[TARGET][tr.year <= 2005] > 0).mean():.1%} | modern era {(tr[TARGET][tr.year > 2005] > 0).mean():.1%}")
print("Train years:", tr.year.min(), "-", tr.year.max(), "| Test years:", te.year.min(), "-", te.year.max())
print("Test years absent from train:", sorted(set(te.year) - set(tr.year)))

## 7. Exploratory data analysis
**In simple words:** three pictures that shape every later decision: (1) damage is extremely heavy-tailed, (2) train and test cover different years, (3) the *share of rows with damage* is very different in the old and modern reporting eras.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
yy = tr[TARGET]; dmg = yy[yy > 0]
top1 = dmg.sort_values(ascending=False).head(int(len(dmg) * 0.01))
print(f"Zero-damage share: {(yy == 0).mean():.1%}")
print(f"Median / 90th / 99th pct of damaged rows: ${dmg.median():,.0f} / ${dmg.quantile(.9):,.0f} / ${dmg.quantile(.99):,.0f}")
print(f"Top 1% of damaged rows hold {top1.sum() / dmg.sum():.1%} of all dollars")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
ax[0].hist(np.log10(dmg), bins=60, color="#4c78a8"); ax[0].set_title("Damaged rows: log10(dollars)"); ax[0].set_xlabel("log10(damage USD)")
yr = pd.DataFrame({"train": tr.year.value_counts().sort_index(), "test": te.year.value_counts().sort_index()}).fillna(0)
yr.plot.bar(ax=ax[1], width=0.9); ax[1].set_title("Rows per year (train vs test)")
tr.groupby("year")[TARGET].apply(lambda s: (s > 0).mean()).plot(ax=ax[2], marker="o", color="#f58518")
ax[2].set_title("Share of rows with damage, by year (train)")
plt.tight_layout(); plt.show()

**What the plots tell us**
- Damage is extremely heavy-tailed → a few huge events dominate the dollar metric, so we must not shrink predictions toward the mean (→ tail-weighted regressor + dollar scale factor).
- Years 1997-2005 look like a different *reporting regime* (most rows have damage) than 2007-2016 → an `old` flag, era-specific thresholds, and re-weighting when tuning.
- Test contains years that train never saw (1993-95, 2017-2025) → `year` is clipped to the train range, and the `old` flag also covers 1993-95.

## 8. Feature engineering — frozen text embeddings 📖
**In simple words:** a pretrained language model (`all-MiniLM-L6-v2`) reads each narrative and returns 384 numbers capturing its meaning ("roof torn off" ≈ "roof ripped away"). Here the model is **frozen** — we only use it to read.

- The event text is prefixed with its event type (`"Hail: ..."`), so the embedding knows what kind of event it describes.
- Every unique text is encoded only once and copied back to its rows.
- The 384 numbers are squeezed with PCA to 32 (event) / 24 (episode). PCA is fitted on **train texts only, no labels**.
- A *missing* episode narrative stays `NaN` (not zero) — "no narrative" is information (it marks the old era).

In [5]:
enc = SentenceTransformer(CFG["ENCODER"], device=device)
if device == "cuda": enc.half()

def embed_unique(texts, max_len, bs=512):
    enc.max_seq_length = max_len
    uniq = pd.unique(pd.Series(texts))
    emb = enc.encode(list(uniq), batch_size=bs, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    idx = pd.Series(np.arange(len(uniq)), index=uniq)
    return emb.astype(np.float32), idx.loc[pd.Series(texts)].values

def ev_text(d):
    return (d.event_type.astype(str) + ": " + d.event_narrative_redacted.fillna("").astype(str).str.strip()).tolist()

all_ev = ev_text(tr) + ev_text(te)
ev_u, ev_i = embed_unique(all_ev, CFG["EV_LEN"]); tick(f"event embeddings: {len(ev_u):,} unique of {len(all_ev):,}")
pca_ev = PCA(CFG["PCA_EV"], random_state=0).fit(ev_u[np.unique(ev_i[:ntr])])
EV = pca_ev.transform(ev_u).astype(np.float32)[ev_i]; del ev_u

ep_all = pd.concat([tr.episode_narrative_redacted, te.episode_narrative_redacted], ignore_index=True)
ep_has = ep_all.notna().values
ep_u, ep_i = embed_unique(ep_all[ep_has].astype(str).str.strip().tolist(), CFG["EP_LEN"], bs=256)
n_tr_has = int(ep_has[:ntr].sum())
pca_ep = PCA(CFG["PCA_EP"], random_state=0).fit(ep_u[np.unique(ep_i[:n_tr_has])])
EP = np.full((len(ep_all), CFG["PCA_EP"]), np.nan, np.float32)      # missing episode narrative -> NaN (not zero)
EP[ep_has] = pca_ep.transform(ep_u).astype(np.float32)[ep_i]; del ep_u
del enc; gc.collect(); torch.cuda.empty_cache()
tick(f"episode embeddings done; EV {EV.shape} EP {EP.shape}")

[  2.0 min] event embeddings: 532,076 unique of 588,941
[  2.7 min] episode embeddings done; EV (588941, 32) EP (588941, 24)

## 9. Feature engineering — structured fields
**In simple words:** convert the table columns into clean numeric clues. Categories stay categories (LightGBM handles them natively). Blank injuries/deaths become 0, but blank magnitude stays `NaN` — a missing measurement is not the same as zero.

Era clues: `year` (clipped to 1997-2016), an `old` flag (≤ 2005, also covers 1993-95), narrative lengths and an "episode narrative missing" flag. The category levels are built from train ∪ test (labels not involved) so train and test share the same encoding.

In [6]:
cats = ["state", "event_type", "magnitude_type", "tor_f_scale"]
levels = {c: sorted(set(tr[c].astype(str)) | set(te[c].astype(str))) for c in cats}

def struct(d):
    o = pd.DataFrame(index=d.index)
    o["year"] = d.year.clip(1997, 2016)            # test years outside the train range are clipped
    o["old"] = (d.year <= 2005).astype(int)        # reporting-era flag (also covers 1993-95)
    o["month"] = pd.to_datetime(d.month_name, format="%B").dt.month
    for c in cats: o[c] = pd.Categorical(d[c].astype(str), categories=levels[c])
    for c in ["lon", "lat", "magnitude", "tor_length", "tor_width"]: o[c] = d[c].astype(float)
    o["inj"] = d.injuries_direct.fillna(0) + d.injuries_indirect.fillna(0)
    o["dth"] = d.deaths_direct.fillna(0) + d.deaths_indirect.fillna(0)
    o["ev_len"] = d.event_narrative_redacted.fillna("").str.len()
    o["epi_len"] = d.episode_narrative_redacted.fillna("").str.len()
    o["epi_na"] = d.episode_narrative_redacted.isna().astype(int)
    if CFG["USE_GROUP_SIZE"]:
        o["gsize"] = d.groupby("episode_group").episode_group.transform("size")
    return o

def assemble(S, ev, ep):
    return pd.concat([S.reset_index(drop=True),
                      pd.DataFrame(ev, columns=[f"ev{i}" for i in range(ev.shape[1])]),
                      pd.DataFrame(ep, columns=[f"ep{i}" for i in range(ep.shape[1])])], axis=1)

S_tr, S_te = struct(tr), struct(te)
X_tr = assemble(S_tr, EV[:ntr], EP[:ntr]); X_te = assemble(S_te, EV[ntr:], EP[ntr:])
y = tr.damage_property_usd.values.astype(float); pos = y > 0
old_tr = (tr.year <= 2005).values; old_te = (te.year <= 2005).values
tick(f"features X_tr {X_tr.shape}, X_te {X_te.shape}; train pos-rate old {pos[old_tr].mean():.2f} / modern {pos[~old_tr].mean():.2f}")

[  2.7 min] features X_tr (358976, 73), X_te (229965, 73); train pos-rate old 0.86 / modern 0.36

## 10. The competition metric (re-implemented) and era re-weighting
**In simple words:** Kaggle's score isn't in any library, so we re-implement it exactly as on the Evaluation page.

Test has ~6× the old-era share that train has (17.3% vs 3.0%). If we tuned thresholds on plain OOF predictions, the modern era would dominate. So every row gets a **weight**: old-era rows count `OLD_W ≈ 6.8×` more, which makes the tuning set look like the test set. The same weights go into the metric (`comp_score(..., w)`).

Helper functions defined here:
- `tune_decision` — grid-search *two thresholds (old / modern era) + one global dollar scale* maximizing the weighted score.
- `make_pred` — apply them: probability ≤ threshold → exactly `0.0`, else `expm1(log-pred) × scale`.
- `write_submission` — validate and save the file.

In [7]:
OLD_W = (old_te.mean() / old_tr.mean()) * ((1 - old_tr.mean()) / (1 - old_te.mean()))
w_row = np.where(old_tr, OLD_W, 1.0)
print(f"old-era weight in tuning: {OLD_W:.2f}  (train old share {old_tr.mean():.3f}, test old share {old_te.mean():.3f})")
MAXPRED = 1e10

def comp_score(yv, pred, w=None):
    w = np.ones(len(yv)) if w is None else w
    t, q = yv > 0, pred > 0
    sens = (w * (t & q)).sum() / (w * t).sum(); spec = (w * (~t & ~q)).sum() / (w * ~t).sum()
    occ = max(0.0, sens + spec - 1.0)
    mag = (w[t] * np.minimum(pred[t], yv[t])).sum() / (w[t] * np.maximum(pred[t], yv[t])).sum()
    return 1 + 99 * (0.5 * occ + 0.5 * mag), occ, mag

THR_OLD_GRID = np.arange(0.40, 0.96, 0.05); THR_MOD_GRID = np.arange(0.20, 0.62, 0.03); SCALE_GRID = [2, 3, 4, 6, 8, 10]

def tune_decision(yd, pd_, lmd, oldd, wd):
    """Pick era thresholds + global magnitude scale maximising the (era-weighted) competition score."""
    mag_raw = np.expm1(np.clip(lmd, 0, np.log1p(5e9)))
    best = (-1, None)
    for sc in SCALE_GRID:
        m = np.clip(mag_raw * sc, 0, MAXPRED)
        for to in THR_OLD_GRID:
            for tm in THR_MOD_GRID:
                s_ = comp_score(yd, np.where(pd_ > np.where(oldd, to, tm), m, 0.0), wd)[0]
                if s_ > best[0]: best = (s_, (to, tm, sc))
    return best

def make_pred(p, lm, old, params):
    to, tm, sc = params
    m = np.clip(np.expm1(np.clip(lm, 0, np.log1p(5e9))) * sc, 0, MAXPRED)
    out = np.where(p > np.where(old, to, tm), m, 0.0)
    return np.where(np.isfinite(out), out, 0.0)

def report(name, yd, pred, oldd, wd, yrs):
    print(f"--- {name}")
    print("weighted ALL:", np.round(comp_score(yd, pred, wd), 3), "| OLD:", np.round(comp_score(yd[oldd], pred[oldd]), 3), "| MODERN:", np.round(comp_score(yd[~oldd], pred[~oldd]), 3))
    per = {int(v): round(comp_score(yd[yrs == v], pred[yrs == v])[0], 1) for v in np.unique(yrs) if (yrs == v).sum() > 500 and (yd[yrs == v] > 0).any()}
    print("score by year (spread = how stable it is across years):", per)

def write_submission(pred, tag):
    sub = pd.DataFrame({"sample_id": te.sample_id, "predicted_damage": pred})
    assert sub.sample_id.is_unique and len(sub) == len(te) and np.isfinite(pred).all() and (pred >= 0).all()
    sub.to_csv("submission.csv", index=False)
    print(f"[{tag}] wrote submission.csv: nonzero share {np.mean(pred > 0):.3f} (train pos-rate {pos.mean():.3f}); old-era test nonzero {np.mean(pred[old_te] > 0):.3f}; "
          f"median>0 {np.median(pred[pred > 0]):,.0f}; max {pred.max():,.0f}")

folds = list(GroupKFold(CFG["N_FOLDS"]).split(X_tr, y, tr.episode_group))

old-era weight in tuning: 6.79  (train old share 0.030, test old share 0.173)

**Metric unit tests** — easy cases where we know the answer: all zeros must score 1, a perfect prediction 100, and "2× the truth" must give magnitude 0.5. We also assert the model inputs never contain forbidden columns.

In [ ]:
assert abs(comp_score(y, np.zeros(ntr))[0] - 1) < 1e-9
assert abs(comp_score(y, y)[0] - 100) < 1e-9
assert abs(comp_score(y, 2 * y)[2] - 0.5) < 1e-9
for df_, nm in [(X_tr, "X_tr"), (X_te, "X_te")]:
    bad = {"sample_id", "episode_group", TARGET} & set(df_.columns)
    assert not bad, f"{nm} contains forbidden columns: {bad}"
print("✅ metric unit tests passed | no id / group / label columns among the features")
print("Folds (grouped by episode_group):", [len(b) for _, b in folds], "rows per validation fold")

## 11. Model A — LightGBM on frozen embeddings + structured fields
**In simple words:** for each of 3 folds (storms/episodes never split across folds) we train two gradient-boosting models:
1. **Classifier** → probability that damage > 0.
2. **Regressor** → `log(1 + damage)`, trained **only on rows that really had damage**, with sample weight `damage^0.2` so big events count more.

Both stop early when they stop improving. Out-of-fold (OOF) predictions are saved for honest tuning; test predictions are the average of the fold models. A time guard stops the GBDT stage if it would use more than ~45% of the budget.

Right after tuning we write a **GBDT-only `submission.csv` as a safety net**, so a later failure can never leave us without a valid file.

In [9]:
P_COMMON = dict(n_estimators=600, learning_rate=0.10, num_leaves=63, min_child_samples=50, subsample=0.7, subsample_freq=1,
                colsample_bytree=0.5, reg_lambda=1.0, max_bin=63, random_state=CFG["SEED"], n_jobs=-1, verbose=-1)
oof_p = np.zeros(ntr); oof_lm = np.zeros(ntr); done = np.zeros(ntr, bool)
te_p = np.zeros(len(te)); te_lm = np.zeros(len(te)); n_done = 0; fold_secs = []
for k, (a, b) in enumerate(folds):
    est = (max(fold_secs) / 60) if fold_secs else 0
    if k > 0 and elapsed_min() + est * 1.15 > 0.45 * CFG["BUDGET_MIN"]:      # GBDT may use at most ~45% of the budget
        print(f"time budget: stopping GBDT after {k} folds"); break
    t0 = time.time()
    clf = lgb.LGBMClassifier(**P_COMMON)
    clf.fit(X_tr.iloc[a], pos[a].astype(int), eval_set=[(X_tr.iloc[b], pos[b].astype(int))], callbacks=[lgb.early_stopping(30, verbose=False)])
    ap, bp = a[pos[a]], b[pos[b]]
    reg = lgb.LGBMRegressor(**P_COMMON)
    reg.fit(X_tr.iloc[ap], np.log1p(y[ap]), sample_weight=y[ap] ** CFG["TAIL_ALPHA"],
            eval_set=[(X_tr.iloc[bp], np.log1p(y[bp]))], eval_sample_weight=[y[bp] ** CFG["TAIL_ALPHA"]],
            callbacks=[lgb.early_stopping(30, verbose=False)])
    oof_p[b] = clf.predict_proba(X_tr.iloc[b])[:, 1]; oof_lm[b] = reg.predict(X_tr.iloc[b]); done[b] = True
    te_p += clf.predict_proba(X_te)[:, 1]; te_lm += reg.predict(X_te); n_done += 1
    fold_secs.append(time.time() - t0)
    tick(f"GBDT fold {k}: clf iters {clf.best_iteration_}, reg iters {reg.best_iteration_}, {fold_secs[-1]:.0f}s")
te_p /= n_done; te_lm /= n_done
print(f"GBDT folds completed: {n_done}/{CFG['N_FOLDS']}  (OOF rows covered: {done.mean():.0%})")

yrs_tr = tr.year.values
d = done
best_g = tune_decision(y[d], oof_p[d], oof_lm[d], old_tr[d], w_row[d])
PARAMS_G = best_g[1]
print(f"GBDT-only: best weighted OOF score {best_g[0]:.2f}; thr_old={PARAMS_G[0]:.2f} thr_mod={PARAMS_G[1]:.2f} scale={PARAMS_G[2]}")
report("GBDT only (OOF)", y[d], make_pred(oof_p[d], oof_lm[d], old_tr[d], PARAMS_G), old_tr[d], w_row[d], yrs_tr[d])
write_submission(make_pred(te_p, te_lm, old_te, PARAMS_G), "GBDT-only safety submission")
FINAL = dict(p=te_p, lm=te_lm, params=PARAMS_G)

[  3.3 min] GBDT fold 0: clf iters 585, reg iters 235, 38s
[  3.9 min] GBDT fold 1: clf iters 597, reg iters 149, 37s
[  4.6 min] GBDT fold 2: clf iters 599, reg iters 236, 40s
GBDT folds completed: 3/3  (OOF rows covered: 100%)
GBDT-only: best weighted OOF score 48.84; thr_old=0.45 thr_mod=0.41 scale=4
--- GBDT only (OOF)
weighted ALL: [48.838  0.82   0.146] | OLD: [40.929  0.651  0.156] | MODERN: [47.641  0.8    0.143]
score by year (spread = how stable it is across years): {1997: 25.9, 1998: 46.1, 1999: 51.4, 2000: 46.5, 2001: 39.3, 2002: 43.5, 2003: 36.5, 2004: 41.7, 2005: 40.7, 2007: 46.2, 2008: 46.1, 2009: 47.6, 2010: 47.3, 2011: 51.0, 2012: 48.1, 2013: 47.6, 2014: 53.0, 2015: 45.2, 2016: 44.3}
[GBDT-only safety submission] wrote submission.csv: nonzero share 0.453 (train pos-rate 0.374); old-era test nonzero 0.869; median>0 31,411; max 870,186,827

## 12. Model B — fine-tuned MiniLM (time-boxed per fold)
**In simple words:** the frozen encoder + PCA throws away detail. Here we **fine-tune** the same MiniLM end-to-end so it learns what in the text signals money.

- **Input:** a text pair. Segment 1 = `event type | state | year | magnitude | injuries/deaths | event narrative`; segment 2 = the episode narrative (or `"no episode narrative"`).
- **Two heads** on the mean-pooled embedding: an occurrence logit (BCE loss) and a log-magnitude (Huber loss, damaged rows only, tail-weighted `damage^0.2`).
- **Time-boxed:** each fold trains for a fixed 480 s wall-clock; the learning rate decays linearly over that budget, so training ends already annealed. Mixed precision (fp16) keeps it fast.
- Wrapped in `try/except` — if anything fails, the GBDT-only submission already on disk stays valid.

In [10]:
nn_p = np.zeros(ntr); nn_lm = np.zeros(ntr); nn_done = np.zeros(ntr, bool)
nn_te_p = np.zeros(len(te)); nn_te_lm = np.zeros(len(te)); nn_n = 0

def nn_texts(d):
    mag = np.where(d.magnitude.notna(), d.magnitude.round(2).astype(str) + " " + d.magnitude_type.astype(str), "")
    inj = d.injuries_direct.fillna(0).astype(int) + d.injuries_indirect.fillna(0).astype(int)
    dth = d.deaths_direct.fillna(0).astype(int) + d.deaths_indirect.fillna(0).astype(int)
    first = (d.event_type.astype(str) + " | " + d.state.astype(str) + " | " + d.year.astype(str) + " | " + pd.Series(mag, index=d.index) +
             " | injuries " + inj.astype(str) + " deaths " + dth.astype(str) + " | " + d.event_narrative_redacted.fillna("").astype(str).str.strip())
    second = d.episode_narrative_redacted.fillna("no episode narrative").astype(str).str.strip()
    return first.tolist(), second.tolist()

def run_nn_fold(a, b):
    """Train on rows a, predict rows b and the test set. Returns (p_b, lm_b, p_te, lm_te)."""
    import torch.nn as nn
    ids_all = NN_IDS
    pad = NN_PAD
    def collate(idx):
        L = max(len(ids_all[i]) for i in idx)
        arr = np.full((len(idx), L), pad, np.int64); att = np.zeros((len(idx), L), np.int64)
        for j, i in enumerate(idx):
            n = len(ids_all[i]); arr[j, :n] = ids_all[i]; att[j, :n] = 1
        return torch.from_numpy(arr).to("cuda", non_blocking=True), torch.from_numpy(att).to("cuda", non_blocking=True)

    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.enc = AutoModel.from_pretrained(CFG["ENCODER"]); h = self.enc.config.hidden_size
            self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(h, 128), nn.GELU(), nn.Linear(128, 2))
        def forward(self, ids, att):
            o = self.enc(input_ids=ids, attention_mask=att).last_hidden_state
            m = att.unsqueeze(-1).to(o.dtype); pooled = (o * m).sum(1) / m.sum(1).clamp(min=1)
            return self.head(pooled).float()

    model = Net().cuda()
    opt = torch.optim.AdamW([{"params": model.enc.parameters(), "lr": CFG["NN_LR"]}, {"params": model.head.parameters(), "lr": 1e-3}], weight_decay=0.01)
    base_lrs = [g["lr"] for g in opt.param_groups]
    scaler = torch.cuda.amp.GradScaler()
    bce = nn.BCEWithLogitsLoss(); hub = nn.SmoothL1Loss(reduction="none")
    tgt_p = torch.tensor(pos.astype(np.float32)); tgt_lm = torch.tensor((np.log1p(y) / 10).astype(np.float32))
    wgt = torch.tensor((np.where(pos, y, 1.0) ** CFG["TAIL_ALPHA"]).astype(np.float32)); wgt = wgt / wgt[torch.tensor(pos)].mean()
    pos_t = torch.tensor(pos)
    T = CFG["NN_FOLD_TRAIN_SEC"]; t0 = time.time(); step = 0; bs = CFG["NN_BS"]; model.train(); stop = False; ep = 0
    while not stop:
        perm = np.random.permutation(a); ep += 1
        for s in range(0, len(perm) - bs + 1, bs):
            frac = (time.time() - t0) / T
            if frac >= 1: stop = True; break
            mult = min(1.0, (step + 1) / 200) * max(0.02, 1 - frac)
            for g, bl in zip(opt.param_groups, base_lrs): g["lr"] = bl * mult
            idx = perm[s:s + bs]; ids, att = collate(idx)
            ti = torch.from_numpy(idx)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                out = model(ids, att)
            lp = bce(out[:, 0], tgt_p[ti].cuda())
            pm = pos_t[ti].cuda().float()
            lr_ = (hub(out[:, 1], tgt_lm[ti].cuda()) * wgt[ti].cuda() * pm).sum() / pm.sum().clamp(min=1)
            loss = lp + 3.0 * lr_
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); step += 1
            if step % 300 == 0: print(f"   nn step {step} ep {ep} loss {loss.item():.4f} ({time.time()-t0:.0f}s)", flush=True)
    print(f"   nn trained {step} steps (~{step*bs/len(a):.2f} epochs) in {time.time()-t0:.0f}s")
    model.eval()
    def predict(rows):
        rows = np.asarray(rows); order = rows[np.argsort([len(ids_all[i]) for i in rows])]
        res = {}; P = np.zeros(len(rows)); M = np.zeros(len(rows)); outs = []
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
            for s in range(0, len(order), 512):
                idx = order[s:s + 512]; ids, att = collate(idx); o = model(ids, att)
                outs.append(torch.cat([torch.sigmoid(o[:, :1]), o[:, 1:] * 10], 1).cpu().numpy())
        o = np.concatenate(outs); pos_in_rows = {r: j for j, r in enumerate(rows)}
        pm_ = np.zeros((len(rows), 2)); pm_[[pos_in_rows[r] for r in order]] = o
        return pm_[:, 0], pm_[:, 1]
    pb, lb = predict(b); pt, lt = predict(ntr + np.arange(len(te)))
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return pb, lb, pt, lt

def fake_nn_fold(a, b):   # offline smoke-test stand-in only
    r = np.random.default_rng(0)
    return (np.clip(oof_p[b] + r.normal(0, .15, len(b)), 0, 1), oof_lm[b] + r.normal(0, .8, len(b)),
            np.clip(te_p + r.normal(0, .15, len(te)), 0, 1), te_lm + r.normal(0, .8, len(te)))

if CFG["NN_ENABLE"]:
    try:
        if not CFG["NN_FAKE"]:
            from transformers import AutoTokenizer, AutoModel
            tok = AutoTokenizer.from_pretrained(CFG["ENCODER"]); NN_PAD = tok.pad_token_id
            f1, f2 = nn_texts(tr); g1, g2 = nn_texts(te)
            t0 = time.time()
            NN_IDS = tok(f1 + g1, f2 + g2, truncation=True, max_length=CFG["NN_MAXLEN"])["input_ids"]
            tick(f"tokenised {len(NN_IDS):,} pairs in {time.time()-t0:.0f}s; mean len {np.mean([len(x) for x in NN_IDS[:20000]]):.0f}")
        per_fold_min = CFG["NN_FOLD_TRAIN_SEC"] / 60 + 3.5       # training + (generous) inference/model-load allowance
        for k, (a, b) in enumerate(folds):
            if not done[b].all(): continue
            if elapsed_min() + per_fold_min > CFG["BUDGET_MIN"] - 4:
                print(f"time budget: skipping NN fold {k} and later (elapsed {elapsed_min():.1f} min)"); break
            t0 = time.time()
            pb, lb, pt, lt = (fake_nn_fold if CFG["NN_FAKE"] else run_nn_fold)(a, b)
            nn_p[b], nn_lm[b], nn_done[b] = pb, lb, True
            nn_te_p += pt; nn_te_lm += lt; nn_n += 1
            tick(f"NN fold {k} done in {(time.time()-t0)/60:.1f} min")
    except Exception:
        print("!!! NN stage failed; keeping the GBDT-only submission already written. Traceback:"); traceback.print_exc()

[  7.6 min] tokenised 588,941 pairs in 153s; mean len 85
   nn step 300 ep 1 loss 0.3065 (30s)
   nn step 600 ep 1 loss 0.2961 (61s)
   nn step 900 ep 1 loss 0.2491 (92s)
   nn step 1200 ep 1 loss 0.2715 (123s)
   nn step 1500 ep 1 loss 0.2800 (154s)
   nn step 1800 ep 1 loss 0.2800 (185s)
   nn step 2100 ep 2 loss 0.2380 (216s)
   nn step 2400 ep 2 loss 0.2276 (247s)
   nn step 2700 ep 2 loss 0.1998 (278s)
   nn step 3000 ep 2 loss 0.2522 (309s)
   nn step 3300 ep 2 loss 0.2486 (340s)
   nn step 3600 ep 2 loss 0.2058 (372s)
   nn step 3900 ep 3 loss 0.2046 (403s)
   nn step 4200 ep 3 loss 0.2683 (434s)
   nn step 4500 ep 3 loss 0.2195 (465s)
   nn trained 4646 steps (~2.48 epochs) in 480s
[ 16.8 min] NN fold 0 done in 9.2 min
   nn step 300 ep 1 loss 0.3880 (31s)
   nn step 600 ep 1 loss 0.3496 (62s)
   nn step 900 ep 1 loss 0.1886 (93s)
   nn step 1200 ep 1 loss 0.3406 (124s)
   nn step 1500 ep 1 loss 0.2763 (156s)
   nn step 1800 ep 1 loss 0.2466 (187s)
   nn step 2100 ep 2 loss 0.2

## 13. Blend both models out-of-fold, re-tune decisions, write the final submission
**In simple words:** the two models make different mistakes, so averaging helps. We pick a blend weight **separately for each head**:
- **Occurrence weight (WP):** the one whose best-threshold balanced accuracy is highest (era-weighted).
- **Magnitude weight (WM):** the one with the lowest tail-weighted log-MSE.

Then we re-tune thresholds + scale on the blended OOF predictions. The blend replaces the safety submission **only if it beats GBDT-only by more than 0.1 points** on the same rows.

In [11]:
if nn_n > 0:
    nn_te_p /= nn_n; nn_te_lm /= nn_n
    ok = done & nn_done
    yd, oldd, wd, yr_ = y[ok], old_tr[ok], w_row[ok], yrs_tr[ok]
    gp, glm, np_, nlm = oof_p[ok], oof_lm[ok], nn_p[ok], nn_lm[ok]

    def bal_skill(yv, pv, t, w):
        tt = yv > 0; q = pv > t
        return (w * (tt & q)).sum() / (w * tt).sum() + (w * (~tt & ~q)).sum() / (w * ~tt).sum() - 1
    def occ_quality(pv):
        tot = 0.0
        for msk, grid in [(oldd, THR_OLD_GRID), (~oldd, THR_MOD_GRID)]:
            tot += max(bal_skill(yd[msk], pv[msk], t, wd[msk]) for t in grid) * wd[msk].sum()
        return tot / wd.sum()
    grid_w = [0, 0.2, 0.35, 0.5, 0.65, 0.8, 1.0]
    occ_scores = {wv: occ_quality((1 - wv) * gp + wv * np_) for wv in grid_w}
    WP = max(occ_scores, key=occ_scores.get)
    dm = yd > 0; lw = yd[dm] ** 0.3
    mse = {wv: float((lw * (((1 - wv) * glm[dm] + wv * nlm[dm]) - np.log1p(yd[dm])) ** 2).sum() / lw.sum()) for wv in grid_w}
    WM = min(mse, key=mse.get)
    print("occurrence quality by NN weight:", {k_: round(v, 4) for k_, v in occ_scores.items()}, "-> WP =", WP)
    print("weighted log-MSE by NN weight  :", {k_: round(v, 4) for k_, v in mse.items()}, "-> WM =", WM)

    bp_, blm_ = (1 - WP) * gp + WP * np_, (1 - WM) * glm + WM * nlm
    best_b = tune_decision(yd, bp_, blm_, oldd, wd); PARAMS_B = best_b[1]
    g_only = tune_decision(yd, gp, glm, oldd, wd)[0]
    print(f"on the same rows: GBDT-only {g_only:.2f} vs blend {best_b[0]:.2f}; thr_old={PARAMS_B[0]:.2f} thr_mod={PARAMS_B[1]:.2f} scale={PARAMS_B[2]}")
    report("BLEND (OOF)", yd, make_pred(bp_, blm_, oldd, PARAMS_B), oldd, wd, yr_)
    if best_b[0] > g_only + 0.1:
        tp, tlm = (1 - WP) * te_p + WP * nn_te_p, (1 - WM) * te_lm + WM * nn_te_lm
        write_submission(make_pred(tp, tlm, old_te, PARAMS_B), "BLEND final")
        FINAL_MODE = "blend"
    else:
        print("blend did not beat GBDT-only out-of-fold; keeping the GBDT-only submission.")
else:
    print("NN stage produced no folds; final submission = GBDT-only (already written).")
tick("DONE (limit 90 min)")

occurrence quality by NN weight: {0: 0.798, 0.2: 0.8124, 0.35: 0.8196, 0.5: 0.8212, 0.65: 0.8172, 0.8: 0.8113, 1.0: 0.8017} -> WP = 0.5
weighted log-MSE by NN weight  : {0: 3.3389, 0.2: 3.05, 0.35: 2.9145, 0.5: 2.8487, 0.65: 2.8524, 0.8: 2.9257, 1.0: 3.1317} -> WM = 0.5
on the same rows: GBDT-only 48.84 vs blend 50.95; thr_old=0.45 thr_mod=0.44 scale=3
--- BLEND (OOF)
weighted ALL: [50.954  0.842  0.167] | OLD: [43.544  0.68   0.179] | MODERN: [49.817  0.824  0.162]
score by year (spread = how stable it is across years): {1997: 29.6, 1998: 45.3, 1999: 51.8, 2000: 48.6, 2001: 43.9, 2002: 46.6, 2003: 41.7, 2004: 43.2, 2005: 43.4, 2007: 48.4, 2008: 48.1, 2009: 51.1, 2010: 49.1, 2011: 52.9, 2012: 49.6, 2013: 51.0, 2014: 56.0, 2015: 47.6, 2016: 46.9}
[BLEND final] wrote submission.csv: nonzero share 0.456 (train pos-rate 0.374); old-era test nonzero 0.886; median>0 27,227; max 2,047,247,109
[ 36.0 min] DONE (limit 90 min)

**Reading the blend result**
- The fine-tuned transformer alone is already slightly better than LightGBM at occurrence (quality 0.802 vs 0.798) and at magnitude (log-MSE 3.13 vs 3.34), but a **50/50 mix beats both** on both heads (0.821 and 2.85). That's the classic ensemble effect: two different "readers" of the text make different errors.
- Gain on the era-weighted OOF score: **48.84 → 50.95**.
- ⚠️ These are *in-sample-tuned* numbers: blend weights, thresholds and scale are all chosen on the same OOF predictions (only 2 + 3 free parameters, but still slightly optimistic). The first notebook's nested validation avoids this; treat 50.95 as an upper-ish estimate.

## 14. Diagnostics & error analysis
**In simple words:** a single number hides a lot. Here we rebuild the final out-of-fold predictions, check how much of the big-dollar tail we capture, and read the biggest mistakes. Look for patterns, e.g. huge flood / hurricane / surge events with short, vague narratives.

In [ ]:
if globals().get("FINAL_MODE") == "blend":
    oof_final = np.zeros(ntr); oof_final[ok] = make_pred(bp_, blm_, old_tr[ok], PARAMS_B); covered = ok
else:
    oof_final = make_pred(oof_p, oof_lm, old_tr, PARAMS_G); covered = done

yc, pc = y[covered], oof_final[covered]
t_ = yc > 0
top = yc >= np.quantile(yc[t_], 0.99)
print(f"Top-1% damaged rows: true total ${yc[top].sum():,.0f} | predicted total ${pc[top].sum():,.0f}")
print(f"Dollar capture (Σmin / Σtrue) over all damaged rows: {np.minimum(pc[t_], yc[t_]).sum() / yc[t_].sum():.1%}")

gap = np.where(covered, y - oof_final, -np.inf)
worst = np.argsort(-gap)[:10]
err = tr.iloc[worst][["year", "state", "event_type"]].copy()
err["true_usd"] = y[worst]; err["pred_usd"] = oof_final[worst].round(0)
err["narrative"] = tr.event_narrative_redacted.iloc[worst].fillna("").str.slice(0, 150).values
pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 220)
err

## 15. Final submission check
**In simple words:** re-open the saved `submission.csv` from disk and validate it against every rule on the Submission page (unique ids, same ids as `sample_submission.csv`, finite, non-negative, exact zeros for "no damage").

In [ ]:
sub = pd.read_csv("submission.csv")
assert list(sub.columns) == ["sample_id", "predicted_damage"]
assert len(sub) == len(te) and sub.sample_id.is_unique and set(sub.sample_id) == set(te.sample_id)
assert np.isfinite(sub.predicted_damage).all() and (sub.predicted_damage >= 0).all()
assert (sub.predicted_damage > 0).any(), "everything is zero — something went wrong"
if not CFG["SAMPLE"]:
    ss = pd.read_csv(DATA_DIR / "sample_submission.csv")
    assert set(sub.sample_id) == set(ss.sample_id) and len(sub) == len(ss), "ids differ from sample_submission"
print(f"✅ submission.csv valid | rows {len(sub):,} | predicted-damaged share {(sub.predicted_damage > 0).mean():.1%} (train truth {pos.mean():.1%})")
print("Prediction quantiles among predicted-damaged rows:\n", sub.predicted_damage[sub.predicted_damage > 0].quantile([.1, .5, .9, .99, 1]).round(0).to_string())
sub.head()

## 16. Runtime & compliance report
**In simple words:** the whole pipeline (training included) must fit in 1.5 h on a GPU. We read the stopwatch and tick off the rules we can check automatically.

In [ ]:
total_min = elapsed_min()
LIMIT_MIN = 90
checklist = {
    "Contextual encoders only (frozen + fine-tuned MiniLM; no TF-IDF/BoW/n-grams)": True,
    "Encoder is the permitted all-MiniLM-L6-v2": CFG["ENCODER"].endswith("all-MiniLM-L6-v2"),
    "episode_group / sample_id not model features": not ({"episode_group", "sample_id"} & set(X_tr.columns)),
    "Test placeholder (-1.0) never used for training": bool((te[TARGET] == -1.0).all()) and TARGET not in X_tr.columns and TARGET not in X_te.columns,
    "Group-aware CV (episodes never split across folds)": all(set(tr.episode_group.iloc[a]).isdisjoint(tr.episode_group.iloc[b]) for a, b in folds),
    "Group-size features off (not confirmed as allowed)": not CFG["USE_GROUP_SIZE"],
    "Submission: unique ids, finite, non-negative": True,
    f"Runtime within {LIMIT_MIN} min GPU limit": total_min <= LIMIT_MIN,
}
print(f"Total runtime: {total_min:.1f} min on {device.upper()} | limit {LIMIT_MIN} min | internal budget {CFG['BUDGET_MIN']} min")
for k_, v_ in checklist.items(): print(("✅" if v_ else "❌"), k_)
assert all(checklist.values()), "a compliance check failed"

## 17. Auto-generated SOLUTION.md
**In simple words:** top teams must provide a `SOLUTION.md` write-up. This cell writes it with the real numbers from this run. It also works as a README for GitHub.

In [ ]:
mode = globals().get("FINAL_MODE", "GBDT-only")
if mode == "blend":
    prm = PARAMS_B
    s_, o_, m_ = comp_score(yd, make_pred(bp_, blm_, oldd, PARAMS_B), wd)
    blend_txt = f"Blend weights: occurrence head {WP:.2f} NN / {1-WP:.2f} GBDT; magnitude head {WM:.2f} NN / {1-WM:.2f} GBDT."
else:
    prm = PARAMS_G
    s_, o_, m_ = comp_score(y[done], make_pred(oof_p[done], oof_lm[done], old_tr[done], PARAMS_G), w_row[done])
    blend_txt = "No blend (the transformer stage did not improve the out-of-fold score)."

solution_md = f"""# StormCost — Solution Write-up

## Summary
Frozen-embedding LightGBM (two heads) + fine-tuned MiniLM (two heads), blended out-of-fold, with era-aware,
metric-aware post-processing (per-era decision thresholds + one global dollar scale). Final model: **{mode}**.

**Out-of-fold score (era-weighted to the test mix, thresholds tuned on the same OOF predictions):**
{s_:.2f}  (occurrence {o_:.3f}, magnitude {m_:.3f}). Reference: all-zeros = 1, perfect = 100.
Note: tuning is not nested, so this is slightly optimistic.

## Pipeline
1. **Text (frozen):** `{CFG['ENCODER']}`; event text is prefixed with its event type. Event -> {CFG['PCA_EV']} PCA dims,
   episode -> {CFG['PCA_EP']} PCA dims (PCA fitted on train texts only). Missing episode narrative stays NaN.
2. **Structured features:** year (clipped to 1997-2016), old-era flag (<= 2005), month, lon/lat, magnitude (blank stays NaN),
   magnitude type, tornado fields, event type, state, injuries/deaths totals, narrative lengths, episode-missing flag.
   No episode-group statistics (`USE_GROUP_SIZE={CFG['USE_GROUP_SIZE']}`).
3. **Model A:** LightGBM classifier for P(damage > 0); LightGBM regressor for log1p(damage) on damaged rows,
   sample weight damage^{CFG['TAIL_ALPHA']}. {CFG['N_FOLDS']}-fold GroupKFold by episode; test = average of fold models.
4. **Model B:** fine-tuned MiniLM, text pair (event type | state | year | magnitude | injuries/deaths | event narrative ; episode narrative),
   mean pooling, two heads (BCE + tail-weighted Huber on log-damage), {CFG['NN_FOLD_TRAIN_SEC']} s per fold, linear LR decay.
5. **Blend:** {blend_txt}
6. **Decision rule:** if P > threshold (old era {prm[0]:.2f}, modern era {prm[1]:.2f}) predict expm1(log-pred) x {prm[2]}, otherwise exactly 0.
   Old-era rows are re-weighted x{OLD_W:.2f} during tuning so the tuning set matches the test era mix.

## Rule compliance
Dense contextual encoders only (no TF-IDF / BoW / n-grams); `episode_group` and `sample_id` are not features;
test placeholder never used; public leaderboard never used for tuning; predictions finite, non-negative, exact 0 for "no damage";
no external data. Total runtime {total_min:.1f} min on {device.upper()} (limit {LIMIT_MIN} min).

## Reproducibility
Seed {CFG['SEED']}; all settings in the `CFG` dict; GPU fp16 encoding/training can differ in the last decimals between runs.

## Limitations & ideas
- CV folds are grouped by episode, not by year, so the year shift to the test set (1993-95, 2017-2025) is only approximated
  by the old-era flag and re-weighting.
- Thresholds, scale and blend weights are tuned on the same OOF predictions (few parameters, but not nested).
- The dollar tail is driven by a handful of events; the magnitude score is noisy.
- Ideas: leave-years-out CV, nested tuning, longer NN fine-tuning (only ~36 of 90 min used), larger encoder, more seeds,
  per-event-type scale factors, Tweedie/quantile objectives.
"""
with open("SOLUTION.md", "w") as f: f.write(solution_md)
print(solution_md[:1500], "\n...")

## 18. Limitations & next steps
**Honest limitations**
- The dollar tail is driven by a handful of extreme events (e.g. Hurricane Ike surge, the Joplin tornado); any model's magnitude score on the private leaderboard will be noisy.
- Validation is grouped by *episode*, not by *year*. Test years are disjoint and older-skewed, so OOF scores only approximate the shift (the `old` flag and old-era re-weighting are our patch for it).
- Blend weights, thresholds and the dollar scale are tuned on the same OOF predictions (not nested), so the 50.95 is mildly optimistic.
- Frozen embeddings + PCA lose detail; the fine-tuned model is time-boxed (~2.5 epochs per fold).

**Things to try next (in order of expected value)**
1. Spend the unused runtime (~36 of 90 min used): raise `NN_FOLD_TRAIN_SEC`, or train longer / more folds.
2. Leave-years-out CV + nested tuning for a more trustworthy estimate.
3. Per-event-type scale factors (with a minimum-sample guard) and Tweedie / quantile objectives for the regressor.
4. More seeds, larger `PCA_EV`, a second fine-tuned encoder from the permitted list.
5. Confirm with the hosts whether group-size features are allowed on test, then set `USE_GROUP_SIZE=True`.